# Cattle Re-ID — Etapa 2, Fase 5: Encoder SOURCE (hocico, CMPD300)

Runner de Colab para la **Fase 5**: inspeccionar CMPD300, generar splits, y entrenar el
encoder de hocico (ResNet-50) que después se adapta al dominio de caras. La lógica vive en
`src/`+`scripts/`; este notebook solo orquesta (igual que `colab_runner.ipynb`).

**Las Fases 6+ (harness re-ID, gap crudo, DANN, self-training) son código a construir** — se
suman a este notebook cuando estén.

### Prerrequisitos (una vez)
1. **Commiteá al repo los archivos de Etapa 2** que armamos (la celda 3 verifica que estén):
   - `config.py` con el bloque CMPD300 agregado
   - `src/dataset.py`, `src/train.py`, `src/models.py` actualizados
   - `scripts/00_inspect_cmpd300.py` y `scripts/05_train_source.py` nuevos
2. **Subí CMPD300 a Drive** una vez, como zip: comprimí tu carpeta `datasets/Baseline/` y subila a `MyDrive/cattle_reid/CMPD300_Baseline.zip`.
3. GPU: *Entorno de ejecución → Cambiar tipo → GPU*.

## 0. Verificar GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Sin GPU: Entorno de ejecución -> Cambiar tipo -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}.'
print('DRIVE_ROOT:', DRIVE_ROOT)

## 2. Traer el código (git clone / pull)

In [ ]:
import os
REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}
!git log --oneline -1

## 3. Verificar que el código de Etapa 2 está en el repo

Si algo falta, esta celda te dice qué commitear. No sigas hasta que dé ✅.

In [ ]:
import sys, importlib, dataclasses
sys.path.insert(0, REPO_DIR)
import config; importlib.reload(config)

problems = []
if not hasattr(config, 'CMPD300_DIR'):
    problems.append('config.py sin el bloque CMPD300 (falta CMPD300_DIR / IMAGE_SIZE_S2).')
for f in ['scripts/00_inspect_cmpd300.py', 'scripts/05_train_source.py']:
    if not Path(f).is_file():
        problems.append(f'falta {f}')
try:
    from src.train import RunConfig
    fields = {fl.name for fl in dataclasses.fields(RunConfig)}
    for need in ['data_dir', 'splits_dir', 'num_classes', 'image_size', 'init_from']:
        if need not in fields:
            problems.append(f'RunConfig sin campo "{need}" (src/train.py desactualizado)')
except Exception as e:
    problems.append(f'no pude importar RunConfig: {e}')

if problems:
    print('❌ Falta código de Etapa 2 en el repo:')
    for p in problems: print('   -', p)
    raise SystemExit('Commiteá esos archivos, git push, y volvé a correr (la celda 2 hace pull).')
print('✅ Código de Etapa 2 presente.')

## 4. CMPD300 desde Drive (subir el zip una vez)

Se copia y descomprime a disco local (rápido) y se apunta `config.CMPD300_DIR` ahí vía
variable de entorno (la toman los `!python scripts/...`).

In [ ]:
import zipfile
CMPD_ZIP = DRIVE_ROOT / 'CMPD300_Baseline.zip'
assert CMPD_ZIP.is_file(), (f'No encuentro {CMPD_ZIP}. Comprimí datasets/Baseline/ y subilo a Drive.')

CMPD_LOCAL = Path('/content/cmpd300')
if not CMPD_LOCAL.exists():
    CMPD_LOCAL.mkdir(parents=True)
    print('Descomprimiendo CMPD300...')
    !unzip -q "{CMPD_ZIP}" -d "{CMPD_LOCAL}"
else:
    print('Ya descomprimido en esta sesión.')

# ubicar la carpeta que contiene train/ val/ test/
cand = [CMPD_LOCAL, CMPD_LOCAL / 'Baseline'] + [p for p in CMPD_LOCAL.iterdir() if p.is_dir()]
CMPD_DIR = next((p for p in cand if (p / 'train').is_dir()), None)
assert CMPD_DIR, 'No encuentro train/ dentro del zip. Revisá cómo comprimiste Baseline/.'
os.environ['CMPD300_DATA_DIR'] = str(CMPD_DIR)   # lo toma config._find_cmpd300_dir()
print('CMPD300_DATA_DIR =', CMPD_DIR)
for s in ('train', 'val', 'test'):
    print(f'  {s}: {len(list((CMPD_DIR / s).iterdir()))} carpetas')

## 5. Persistir outputs en Drive (checkpoints / logs / results)

Symlink igual que en `colab_runner.ipynb`: el encoder source (`cmpd300_source.pt`) y los
resultados sobreviven a una desconexión. La Fase 6 va a tomar ese checkpoint de acá.

In [ ]:
import shutil
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
for sub in ('checkpoints', 'logs', 'results'):
    drive_sub = DRIVE_OUTPUTS / sub; drive_sub.mkdir(parents=True, exist_ok=True)
    local_sub = Path(REPO_DIR) / 'outputs' / sub
    if local_sub.exists() and not local_sub.is_symlink():
        for item in local_sub.iterdir():
            tgt = drive_sub / item.name
            if not tgt.exists():
                (shutil.copytree if item.is_dir() else shutil.copy2)(item, tgt)
        shutil.rmtree(local_sub)
    if not local_sub.exists():
        os.symlink(drive_sub, local_sub, target_is_directory=True)
    print(f'outputs/{sub} -> {drive_sub}')

## 6. Fase 5a — Inspección + splits de CMPD300

Genera `outputs/splits_cmpd300/{train,val,test}.json` + `label_map.json`, y reporta clases,
imágenes por split e IDs faltantes en val. No re-splitea: lee las carpetas que ya vienen.

In [ ]:
!python scripts/00_inspect_cmpd300.py

## 7. Smoke test (validar el pipeline antes del run real)

In [ ]:
!python scripts/05_train_source.py --smoke

## 8. Fase 5b — Entrenar el encoder source (real)

ResNet-50, 224 + norm ImageNet, fine-tune completo (`--no-freeze`, mejor para el embedding
que se transfiere). Desde ImageNet por default.

*Warm-start opcional desde tu ResNet de la replicación (Zenodo):* si tenés
`resnet50_backbone.pt` en `outputs/checkpoints/` (persistido en Drive de la Etapa 1), agregá
`--init-from replication`. Ver la advertencia que imprime sobre freeze/normalización.

In [ ]:
!python scripts/05_train_source.py --no-freeze
# warm-start desde Zenodo (si tenés el checkpoint):
# !python scripts/05_train_source.py --no-freeze --init-from replication

## 9. Resultados

In [ ]:
import json
print('=== checkpoints ==='); !ls -lh outputs/checkpoints/ | grep -i cmpd300 || true
p = Path('outputs/results/05_source_summary.json')
if p.exists():
    print('\n=== 05_source_summary.json ===')
    print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False))

## Próximo paso — Fase 6 (a construir)

Con el encoder source entrenado (`outputs/checkpoints/cmpd300_source.pt`), lo que sigue es el
**harness de re-ID**: extractor de embeddings + gallery/probe + Rank-1/mAP, para después medir
el **gap crudo hocico→cara** sobre Ahmed. Ese código lo armamos y se suma acá.